<a href="https://colab.research.google.com/github/Alexis0104343/Analisis_de_datos_2026/blob/main/Data_Analyst_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Analyst Assignment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Introduction

You are working with a US retail customer on a pilot deployment.  They are using technology to track their merchandise throughout their supply chain.  The flow of their supply is:

*   **DC 1:**  Orders are filled and palletized.
*   **Truck:** Pallets travel from the DC 1 to DC 2 via semi-truck.
*   **DC 2:**  Pallets are unloaded, and additional merchandise may be added.  They are then reloaded onto a new truck.
*   **Truck:** Pallets travel from DC 2 to the Store.
*   **Store:** Pallets are unloaded, cases are removed, and stocked, and the empty cases are left behind the building awaiting pickup.

Your job is to dig into the data and find compelling insights to show the value fo the technology and help move the contract from a pilot into a full scaled deployment.



---

## Part 0: Imports

Import necessary packages and

In [ ]:
# YOUR CODE HERE:
import pandas as pd
import plotly.express as px

### Dataset Overview

* Site:  A large space that could contain multiple readers. Ex: DC 1.
* Zone:  Point of interest. These represent areas in which repeaters are installed. These can be thought of as sub-zones.  Ex: Dock Doors.
* Asset ID: The unique ID of the asset.
* Asset Type: The type of thing that is detected (ie tote).
* Device ID: The unique gateway reader ID that detected the device in the zone (there can be multiple in one zone).
* Time est: The time in EST.
* Lon: Longituge
* Lat: Latitude
* Temperature_C / F: Temperature in Celsius, Fahrentheit

## PART 1: Data Overview

### Question 1:

* How many unique cases were we tracking throughout this pilot. (1 pt)
* What are the unique zones we could see (1 pt)
* How many POI's are in each Zone. (2 pts)


In [ ]:
# YOUR CODE HERE:
df_ass1=pd.read_excel("/content/drive/MyDrive/Data/Assignment_1.xlsx")
df_ass1.head()

#df_ass1.info()

,asset_type,asset_id,tag_id,Site,Zone,device_id,time_est,lat,lng,Temperature_C,Temperature_F
0,tote,2,(01)00850027865010(21)0082T0219,DC 1,dock_doors_DC1,7F9A8353E973,2022-08-02 11:59:26.628,47.79158,-65.68902,23.0,73.4
1,tote,2,(01)00850027865010(21)0082T0219,DC 1,dock_doors_DC1,EC5B0499234F,2022-08-02 12:00:22.660,47.79158,-65.68902,23.0,73.4
2,tote,2,(01)00850027865010(21)0082T0219,DC 1,dock_doors_DC1,7F9A8353E973,2022-08-02 12:01:11.234,47.79158,-65.68902,23.0,73.4
3,tote,2,(01)00850027865010(21)0082T0219,DC 1,dock_doors_DC1,3D8B2BDB8673,2022-08-02 13:47:58.172,47.79158,-65.68902,23.5,74.3
4,tote,2,(01)00850027865010(21)0082T0670,DC 1,dock_doors_DC1,7F9A8353E973,2022-08-02 11:58:55.049,47.79158,-65.68902,23.0,73.4


In [ ]:
df_ass1.groupby("Site")["Zone"].nunique().reset_index()

,Site,Zone
0,DC 1,5
1,DC 2,2
2,Store,4
3,Transit,4


### Question 2:

* What is the temperature range we see?  (1pt)
* Where is temperature the highest and lowest (1pt)

In [ ]:
# YOUR CODE HERE:
df_ass1.Temperature_C.max()-df_ass1.Temperature_C.min()
df_ass1.Temperature_C.max()
df_ass1.Temperature_C.min()

19.0

## Part 2: The Journey of a Case

### Question 3:

* Create a visualization that shows where a case was at over time at the zone or POI level. Imagine that this would be included in your presentation to the customer. (Non techical audience) (3 pts)

In [ ]:
# YOUR CODE HERE:
df_2=df_ass1.loc[df_ass1.tag_id=="(01)00850027865010(21)0082T0194"].sort_values(by="time_est").reset_index()
df_clean=df_2.groupby("Site").agg(
    {
        "time_est":["min","max"]

    }
).reset_index().sort_values(by=("time_est","min"))
df_clean

Site                time_est                        
                               min                     max
0     DC 1 2022-08-02 11:58:53.202 2022-08-02 13:50:38.953
2  Transit 2022-08-02 13:51:16.590 2022-08-04 08:34:58.159
1    Store 2022-08-04 08:35:43.433 2022-08-05 13:28:25.543

In [ ]:
df_clean.columns=["Site","start","end"]
df_clean

,Site,start,end
0,DC 1,2022-08-02 11:58:53.202,2022-08-02 13:50:38.953
2,Transit,2022-08-02 13:51:16.590,2022-08-04 08:34:58.159
1,Store,2022-08-04 08:35:43.433,2022-08-05 13:28:25.543


In [ ]:
fig_ev=px.timeline(df_clean,
                       x_start="start",
                       x_end="end",
                       y="Site",
                      color="Site",

                       title="Evolucion en cadena de suministro")
fig_ev.show()

### Question 4:

* Visualize how the temperatue changes over time along its journey.  Imagine that this would be included in your presentation to the customer. (Non techical audience) (4 pts)



In [ ]:
# YOUR CODE HERE:

fig=px.line(df_2.,x="time_est",y="Temperature_C",title="Evolucion de la temperatura")
fig.show()

### Question 5:
* Visualize the lon lat data on a map to show how the case traveled.  You may incorporate any other additional information to make this more impactful. Imagine that this would be included in your presentation to the customer. (Non techical audience) (5 pts)

**Do not worry if this looks like non-sense on a map.  Ex:  The trip may appear to occur over water or in a forest because this is a toy dataset.**

In [ ]:
# YOUR CODE HERE:
map=px.scatter_mapbox(df_2,lat="lat",lon="lng",zoom=15,height=500,
                      hover_name="device_id"   ) # Lo que sale al pasar el mouse

                                # Color según el país)
map.update_layout(mapbox_style="open-street-map")
map.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
map.show()

# Part 3: Customer Questions


### Question 6:

The customer wants to understand the efficieny of ther DC operations.
* Based on what you see in the data, (all zones except for STORE), which parts of their operation are most & least "efficient? (10 pts)

In [ ]:
# YOUR CODE HERE
df_efic=df_ass1.groupby(["tag_id","Zone","Site"]).agg({"time_est":["min","max"]}).reset_index().sort_values(by=["tag_id",("time_est","min")])
df_efic.columns=["tag_id","Zone","Site","start","end"]
df_efic["lapso"]=df_efic["end"]-df_efic["start"]
df_prom=df_efic.groupby(["Zone","Site"])["lapso"].mean().reset_index().sort_values(by="lapso",ascending=False)
df_prom
# df_efic=df_ass1.groupby(["tag_id","Zone"]).agg({"time_est":["min","max"]}).reset_index().sort_values(by=["tag_id",("time_est","min")])
# df_efic.columns=["tag_id","Zone","start","end"]
# df_efic["lapso"]=df_efic["end"]-df_efic["start"]
# df_efic.groupby("Zone")["lapso"].mean().reset_index().sort_values(by="lapso",ascending=False)


,Zone,Site,lapso
2,PhoneKit1Bridge,Transit,1 days 12:58:12.669323529
5,PhoneKit2GW,Transit,1 days 11:25:57.477382352
4,PhoneKit2Bridge,Transit,1 days 11:23:37.628606060
13,store_back_Store,Store,1 days 03:22:23.532000
10,receiving_Store,Store,1 days 01:52:26.496794117
14,store_front_Store,Store,0 days 15:10:07.511382352
12,staging_DC2,DC 2,0 days 10:53:05.188888888
7,dock_doors_DC2,DC 2,0 days 10:34:23.853200
3,PhoneKit1GW,Transit,0 days 04:58:32.805304347
9,point_of_sale_Store,Store,0 days 02:28:23.318904761


YOUR TEXT ANSWER:
La zona menos eficiente es PhoneKit2Bridge  coorespondiente al Sitio de transito
La mas eficiente es dock_doors_DC1 la cual pertenece al Sitio DC1


### Question 7:

The customer wants to understand the stocking efficiency in stores.
* Based on what you see in the data, how quickly did the store unload and stock the merchandise. (5 pts)
* How could this be converted in a KPI that a regional manager could track?  (5 pts)

In [ ]:
# YOUR CODE HERE
df_3=df_prom.query("Site == 'Store'")
df_3
df_3.lapso.sum()


Timedelta('2 days 22:53:20.859081230')

YOUR TEXT ANSWER HERE
En promedio tardaron 2 dias y 22 horas en ese procesos,en base este promedio puede analizar otras tiendas y cuanto tardan en descargar y almacenar la mercancia ,si es mayor a este registrado,podria existir algun problema.

### Question 8:

Please explain what you would ask for and what you will do with this data, given that you can talk with the following people (no code needed):


YOUR TEXT ANSWER HERE
* a. X(Gerente de logistica) ¿Por que estan tardado el proceso en el sitio PhoneKit2Bridge?
En caso de ser el proceso en como se trabaja podria proponer aumentar los turnos por ejemplo.
* b. Y(Persona escargada de los receptores) ¿ Hay zonas muertas donde los identificadores no detecten los productos?
Podria promoner invertir en identificadores de mayor alcance.

## Part 4: Bonus Insights

### Question 8

The customer is open to hearing about additional insights you found in the data above and beyond what they asked for.
* Based on what you can see in the data, are there any other interesting insights that the customer may want to hear about? (Up to 15 bonus points)



In [ ]:
# YOUR CODE HERE
# Another metrics that could be interesting would be temperature by time

YOUR TEXT ANSWER HERE
